### **Implémentation du notebook pour l’indice de Potentiel Solaire (SEP)**


L’objectif est de construire un modèle capable de prédire le potentiel solaire quotidien (shortwave_radiation_sum ou un indice normalisé) à partir des autres variables météorologiques.
Vous trouverez ci-dessous un notebook complet, structuré et commenté, conforme au jeu de données décrit (87 240 observations, 42 villes, 2020‑2025).

Note sur les variables disponibles :
Le dataset fourni contient shortwave_radiation_sum mais ne mentionne pas explicitement sunshine_duration ni daylight_duration.
Deux approches sont possibles :

Utiliser directement shortwave_radiation_sum comme cible (régression).

Calculer une durée du jour théorique (fonction de la latitude et du jour de l’année) et, si sunshine_duration est absente, l’estimer ou se contenter de la première approche.
Le notebook ci‑dessous adopte l’approche 1 (cible = shortwave_radiation_sum), la plus robuste avec les données disponibles.

# Notebook : Prédiction du Potentiel Solaire (SEP)

## 1. Import des bibliothèques
## 2. Chargement et analyse exploratoire
## 3. Feature engineering
## 4. Préparation des données (train/test temporel, scaling)
## 5. Entraînement et comparaison de modèles
## 6. Sélection et sauvegarde du meilleur modèle
## 7. Interprétabilité (SHAP, importance des features)
## 8. Conclusion

### **2. Import des bibliothèques**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')


from pathlib import Path

DATASET_PATH = Path("C:/Users/LENOVO/Desktop/hackathon/HACKATHON-INDABAX-CAMEROON-2026-main/data/Dataset_complet_Meteo.csv")


: 

### **3. Chargement et analyse exploratoire**

In [ ]:
# Chargement
df = pd.read_csv(DATASET_PATH, parse_dates=['time'])
df.set_index('time', inplace=True)
df.sort_index(inplace=True)

# Aperçu
print(df.shape)
df.head()

# Vérification des valeurs manquantes pour la cible et les features clés
target = 'shortwave_radiation_sum'
features_candidates = ['temperature_2m_mean', 'relative_humidity_2m_mean', 
                       'wind_speed_10m_max', 'precipitation_sum', 
                       'et0_fao_evapotranspiration']

print(df[features_candidates + [target]].isnull().sum())

# Distribution de la cible
plt.figure(figsize=(10,4))
sns.histplot(df[target], bins=50, kde=True)
plt.title('Distribution du rayonnement solaire quotidien (MJ/m²)')
plt.show()

# Série temporelle pour quelques villes
cities_sample = df['city'].unique()[:5]
plt.figure(figsize=(12,6))
for city in cities_sample:
    subset = df[df['city'] == city].sort_index()
    plt.plot(subset.index, subset[target], label=city, alpha=0.7)
plt.legend()
plt.title('Évolution temporelle du rayonnement solaire')
plt.show()

# Corrélation avec les autres variables
corr = df[features_candidates + [target]].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Matrice de corrélation')
plt.show()

### **Interprétation :**

Le rayonnement solaire est généralement plus élevé en saison sèche et dans les régions nordiques.

Forte corrélation positive avec la température et l’évapotranspiration, négative avec l’humidité et les précipitations.

### **4. Feature engineering**

On crée des caractéristiques temporelles, des rolling windows et des lags pour capturer la saisonnalité et la persistance.

In [ ]:
# 4.1 Caractéristiques temporelles
df['month'] = df.index.month
df['day_of_year'] = df.index.dayofyear
df['sin_doy'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['cos_doy'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
df['is_dry_season'] = df['month'].isin([11,12,1,2,3]).astype(int)

# 4.2 Variables dérivées par ville (rolling et lags)
# Rolling moyen sur 3, 7, 14 jours
for window in [3, 7, 14]:
    df[f'rad_roll{window}'] = df.groupby('city')[target].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )
    df[f'temp_roll{window}'] = df.groupby('city')['temperature_2m_mean'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )
    df[f'hum_roll{window}'] = df.groupby('city')['relative_humidity_2m_mean'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )

# 4.3 Lags (valeur du jour précédent)
df['rad_lag1'] = df.groupby('city')[target].shift(1)
df['temp_lag1'] = df.groupby('city')['temperature_2m_mean'].shift(1)

# 4.4 Interaction : produit température * rayonnement (utile pour modèles linéaires)
df['temp_rad_interact'] = df['temperature_2m_mean'] * df[target]

# 4.5 Supprimer les lignes de début de série où les lags sont NaN
df.dropna(inplace=True)

**Remarque :** Si vous souhaitez calculer une durée du jour théorique (pour normaliser le rayonnement), ajoutez :

In [ ]:
def daylength(lat, doy):
    # Formule simple (radians) : à adapter selon la latitude en degrés
    phi = np.radians(lat)
    delta = -np.radians(23.44) * np.cos(2*np.pi/365 * (doy + 10))
    omega = np.arccos(-np.tan(phi) * np.tan(delta))
    return 24 * omega / np.pi

df['daylight_duration'] = df.apply(lambda r: daylength(r['latitude'], r['day_of_year']), axis=1)
# Puis normaliser : df['SEP_norm'] = df[target] / df['daylight_duration']

### **5. Préparation des données (split temporel et scaling)**

In [ ]:
# Liste des features d'entrée (on exclut la cible et les identifiants)
feature_cols = ['temperature_2m_mean', 'relative_humidity_2m_mean', 
                'wind_speed_10m_max', 'precipitation_sum', 
                'et0_fao_evapotranspiration',
                'month', 'sin_doy', 'cos_doy', 'is_dry_season',
                'rad_roll3', 'rad_roll7', 'rad_roll14',
                'temp_roll3', 'temp_roll7', 'temp_roll14',
                'hum_roll3', 'hum_roll7', 'hum_roll14',
                'rad_lag1', 'temp_lag1', 'temp_rad_interact']

X = df[feature_cols]
y = df[target]

# Split temporel : 2020-2023 pour train, 2024-2025 pour test
train_mask = (df.index.year < 2024)
test_mask = (df.index.year >= 2024)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train : {X_train.shape}, Test : {X_test.shape}")

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### **6. Entraînement et comparaison de modèles**

On utilise une validation croisée temporelle (TimeSeriesSplit) sur la période d’entraînement.

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

models = {
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42)
}

results = {}
best_model = None
best_mae = float('inf')

for name, model in models.items():
    # Validation croisée temporelle
    mae_scores = []
    for train_idx, val_idx in tscv.split(X_train_scaled):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, y_pred))
    mean_mae = np.mean(mae_scores)
    results[name] = mean_mae
    print(f"{name} - MAE moyen sur validation croisée : {mean_mae:.2f} MJ/m²")
    
    # Entraînement final sur tout le train
    model.fit(X_train_scaled, y_train)
    y_pred_test = model.predict(X_test_scaled)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    print(f"  → Test MAE : {test_mae:.2f}\n")
    
    if test_mae < best_mae:
        best_mae = test_mae
        best_model = model
        best_name = name

print(f"\nMeilleur modèle : {best_name} avec MAE test = {best_mae:.2f}")

### **7. Sauvegarde du meilleur modèle et du scaler**

In [ ]:
joblib.dump(best_model, 'best_sep_model.pkl')
joblib.dump(scaler, 'scaler_sep.pkl')

# Sauvegarder aussi la liste des features (utile pour l’inférence)
with open('feature_cols_sep.txt', 'w') as f:
    for col in feature_cols:
        f.write(col + '\n')

### **8. Interprétabilité**

##### **8.1 Importance des features (modèles arborescents)**

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    plt.figure(figsize=(10,6))
    plt.title("Importance des features")
    plt.barh(range(len(indices)), importances[indices], align='center')
    plt.yticks(range(len(indices)), [feature_cols[i] for i in indices])
    plt.gca().invert_yaxis()
    plt.show()

#### **8.2 SHAP pour expliquer les prédictions**

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled[:100])  # 100 premiers échantillons

# Graphique global
shap.summary_plot(shap_values, X_test_scaled[:100], feature_names=feature_cols)

# Explication locale pour une observation
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0], 
                X_test_scaled[0], feature_names=feature_cols, matplotlib=True)

#### **8.3 Analyse des résidus**

In [ ]:
y_pred_test = best_model.predict(X_test_scaled)
residuals = y_test - y_pred_test

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.scatter(y_pred_test, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Prédictions')
plt.ylabel('Résidus')
plt.title('Résidus vs Prédictions')

plt.subplot(1,2,2)
sns.histplot(residuals, bins=50, kde=True)
plt.title('Distribution des résidus')
plt.show()

### **9. Bonus : prédiction d’un SEP normalisé (si sunshine_duration disponible)**

Si le dataset contient sunshine_duration et que vous voulez prédire le ratio sunshine_duration / daylight_duration, adaptez le code :

In [ ]:
# Calcul de la cible normalisée
df['SEP_norm'] = df['sunshine_duration'] / df['daylight_duration']  # ou bien shortwave_radiation_sum / daylight_duration

# Puis utilisez SEP_norm comme y dans les étapes précédentes.

### **10. Conclusion et recommandations**
Le notebook ci-dessus est prêt à être exécuté sur le dataset du hackathon.

Les performances peuvent être améliorées en affinant les hyperparamètres (GridSearchCV avec TimeSeriesSplit).

Pensez à gérer les valeurs manquantes éventuelles (interpolation par ville ou médiane glissante).

Pour une production réelle, ajoutez des tests de robustesse (ex : prédiction sur des années très sèches ou très humides).

Résumé des fichiers produits :

best_sep_model.pkl : modèle final

scaler_sep.pkl : normaliseur

feature_cols_sep.txt : liste des features

Notebook complet avec visualisations et interprétabilité.